# DORA metrics from telemetry you already collect: Where they mislead

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/dora-metrics-opentelemetry-cicd-esql). It runs the
article's ES|QL queries over the lab's data and builds the dashboard.

Requires Elasticsearch and Kibana 9.5+ and a `.env` copied from `.env.example`. To set up and run the lab,
follow `README.md`.

In [ ]:
%pip install -q --disable-pip-version-check -r requirements.txt

## 1. Connect

In [ ]:
import json
import os
import pathlib
import statistics
import subprocess
import sys
import warnings
from datetime import datetime

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()
warnings.filterwarnings("ignore", message="No limit defined")

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"
LAB_START = os.getenv("LAB_START", "2026-10-18T20:00:00Z")
LAB_END = os.getenv("LAB_END", "2026-10-19T01:00:00Z")

RULE_NAME = "checkout failed transaction rate by release"
DATA_VIEW_TITLE = "traces-*.otel-*"
DASHBOARD_ID = "dora-metrics-checkout"
STATE_FILE = pathlib.Path(
    ".notebook-state.json"
)  # what this notebook created, for Clean up
state = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}


def save_state():
    STATE_FILE.write_text(json.dumps(state, indent=1))


es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
    "elastic-api-version": "2023-10-31",
}
TIME_FILTER = {"range": {"@timestamp": {"gte": LAB_START, "lt": LAB_END}}}


def kbn(method, path, **kwargs):
    response = requests.request(
        method, f"{KBN_BASE}{path}", headers=KBN_HEADERS, timeout=120, **kwargs
    )
    response.raise_for_status()
    return response.json() if response.text else {}


def fmt(value):
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.3f}".rstrip("0").rstrip(".")
    if isinstance(value, list):
        return ", ".join(fmt(v) for v in value)
    return str(value)


def esql(query, title=None, show=True):
    """Run an ES|QL query over [LAB_START, LAB_END), print it as a table, return the rows as dicts."""
    response = es.esql.query(query=query, filter=TIME_FILTER)
    columns = [c["name"] for c in response["columns"]]
    rows = [dict(zip(columns, values)) for values in response["values"]]
    if show:
        table = [columns] + [[fmt(v) for v in values] for values in response["values"]]
        widths = [max(len(r[i]) for r in table) for i in range(len(columns))]
        if title:
            print(f"\n{title}")
        for i, r in enumerate(table):
            print("  ".join(cell.ljust(w) for cell, w in zip(r, widths)).rstrip())
            if i == 0:
                print("  ".join("-" * w for w in widths))
    return rows


def ts(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


def minutes(start, end):
    return (
        int((ts(end) - ts(start)).total_seconds() / 6 + 0.5) / 10
    )  # rounded half up to 0.1


print(
    f"elasticsearch {es.info()['version']['number']} | space={SPACE} | window {LAB_START} to {LAB_END}"
)

## 2. Create the data view and the alert rule

Run this before the lab, and pass the rule id it prints to `scenario.py`. Existing objects with the same name are reused.

In [ ]:
views = kbn("GET", "/api/data_views")["data_view"]
view = next((v for v in views if v["title"] == DATA_VIEW_TITLE), None)
if view:
    DATA_VIEW_ID = view["id"]
else:
    created = kbn(
        "POST",
        "/api/data_views/data_view",
        json={"data_view": {"title": DATA_VIEW_TITLE, "timeFieldName": "@timestamp"}},
    )
    DATA_VIEW_ID = state["data_view_id"] = created["data_view"]["id"]
    save_state()

found = kbn(
    "GET",
    "/api/alerting/rules/_find",
    params={"search": f'"{RULE_NAME}"', "search_fields": "name", "per_page": 100},
)["data"]
rule = next((r for r in found if r["name"] == RULE_NAME), None)
if not rule:
    body = json.loads(pathlib.Path("rule/custom-threshold-rule.json").read_text())
    body["params"]["searchConfiguration"]["index"] = DATA_VIEW_ID
    rule = kbn("POST", "/api/alerting/rule", json=body)
    state["rule_id"], state["rule_created_at"] = rule["id"], rule["created_at"]
    save_state()

RULE_ID = rule["id"]
print(f"data view {DATA_VIEW_ID} | RULE_ID={RULE_ID}")

## 3. Check the three data sources

After the lab run, set `LAB_START` and `LAB_END` in `.env` to its window and rerun section 1.

In [ ]:
counts = esql(
    """
FROM traces-*.otel-*
| STATS pipeline_spans = COUNT(*) WHERE service.name == "Deliver checkout",
        checkout_spans = COUNT(*) WHERE service.name == "checkout"
""",
    "Pipeline and service spans",
)[0]
if not counts["pipeline_spans"] or not counts["checkout_spans"]:
    raise RuntimeError(
        "No lab data in this window: run the lab (README.md) or fix LAB_START and LAB_END in .env"
    )

esql(
    f"""
FROM .alerts-observability.threshold.alerts-default
| WHERE kibana.alert.rule.name == "{RULE_NAME}"
| KEEP kibana.alert.start, kibana.alert.end, service.version, kibana.alert.case_ids
| SORT kibana.alert.start
""",
    "Alerts",
)

cases = kbn(
    "GET",
    "/api/cases/_find",
    params={
        "owner": "observability",
        "search": f'"{RULE_NAME}"',
        "searchFields": "title",
        "perPage": 100,
        "sortField": "createdAt",
        "sortOrder": "asc",
    },
)["cases"]
cases = [c for c in cases if LAB_START <= c["created_at"] < LAB_END]
modes = {
    i: s["settings"]["index"].get("mode", "standard")
    for i, s in es.indices.get_settings(
        index=".cases", expand_wildcards="all"
    ).body.items()
}
print(f"\n{len(cases)} cases | .cases index mode: {modes}")

try:
    esql(
        """
FROM metrics-githubreceiver.otel-*
| WHERE vcs.change.time_to_merge IS NOT NULL
| STATS branches = COUNT_DISTINCT(vcs.ref.head.name)
""",
        "Pull request metrics",
    )
except Exception as exc:
    print(
        f"\nno GitHub receiver metrics ({type(exc).__name__}): the pull request cells will be skipped"
    )

## 4. Deployment frequency

In [ ]:
rows = esql(
    """
FROM traces-*.otel-*
| WHERE cicd.pipeline.task.type == "deploy"
    AND deployment.environment.name == "production"
    AND cicd.pipeline.task.run.result == "success"
| STATS deployments = COUNT(*),
        versions_deployed = COUNT_DISTINCT(service.version)
    BY hour = BUCKET(@timestamp, 1 hour)
| SORT hour
""",
    "Deployments per hour",
)

In [ ]:
frequency = esql(
    """
FROM traces-*.otel-*
| WHERE service.name == "Deliver checkout"
    AND (kind == "Server" OR cicd.pipeline.task.type == "deploy")
| STATS pipeline_runs = COUNT(*) WHERE kind == "Server",
        failed_runs = COUNT(*) WHERE kind == "Server" AND cicd.pipeline.result != "success",
        deploy_jobs = COUNT(*) WHERE cicd.pipeline.task.type == "deploy",
        deployments = COUNT(*) WHERE cicd.pipeline.task.type == "deploy" AND cicd.pipeline.task.run.result == "success",
        versions_deployed = COUNT_DISTINCT(service.version) WHERE cicd.pipeline.task.type == "deploy" AND cicd.pipeline.task.run.result == "success"
""",
    "Pipeline runs against deployments",
)

## 5. Lead time for changes

In [ ]:
rows = esql(
    """
FROM traces-*.otel-*
| WHERE service.name == "Deliver checkout" AND deployment.environment.name == "production"
| EVAL deploy_end = CASE(
    cicd.pipeline.task.type == "deploy" AND cicd.pipeline.task.run.result == "success",
    TO_DATETIME(TO_LONG(@timestamp) + duration / 1000000))
| STATS committed_at = MAX(TO_DATETIME(github.head_commit.timestamp)),
        deployed_at = MAX(deploy_end),
        attempt = MAX(github.run_attempt)
    BY trace.id, service.version
| WHERE deployed_at IS NOT NULL
| EVAL lead_time_minutes = DATE_DIFF("second", committed_at, deployed_at) / 60.0
| STATS deployments = COUNT(*),
        median_minutes = MEDIAN(lead_time_minutes),
        p90_minutes = PERCENTILE(lead_time_minutes, 90),
        max_minutes = MAX(lead_time_minutes)
    BY rerun = attempt > 1
""",
    "Lead time, merge to production (minutes)",
)

In [ ]:
try:
    esql(
        """
FROM metrics-githubreceiver.otel-*
| WHERE vcs.change.time_to_merge IS NOT NULL
| STATS time_to_merge = MAX(vcs.change.time_to_merge) BY vcs.ref.head.name
| STATS pull_requests = COUNT(*),
        median_minutes = MEDIAN(time_to_merge) / 60.0,
        p90_minutes = PERCENTILE(time_to_merge, 90) / 60.0
""",
        "Pull request time to merge (minutes)",
    )
except Exception as exc:
    print(f"skipped, no GitHub receiver metrics ({type(exc).__name__})")

Pull request open to production: time to merge plus lead time, matched by the last commit message of each branch.

In [ ]:
branches = esql(
    """
FROM traces-*.otel-*
| WHERE service.name == "Deliver checkout" AND github.event == "pull_request" AND kind == "Server"
| KEEP @timestamp, github.head_ref, github.head_commit.message
| SORT @timestamp
""",
    show=False,
)
last_commit = {r["github.head_ref"]: r["github.head_commit.message"] for r in branches}

deployed = {
    r["message"]: r
    for r in esql(
        """
FROM traces-*.otel-*
| WHERE service.name == "Deliver checkout" AND deployment.environment.name == "production"
| EVAL deploy_end = CASE(
    cicd.pipeline.task.type == "deploy" AND cicd.pipeline.task.run.result == "success",
    TO_DATETIME(TO_LONG(@timestamp) + duration / 1000000))
| STATS message = MAX(github.head_commit.message),
        committed_at = MAX(TO_DATETIME(github.head_commit.timestamp)),
        deployed_at = MIN(deploy_end)
    BY service.version
| WHERE deployed_at IS NOT NULL
""",
        show=False,
    )
}

try:
    merge_seconds = {
        r["vcs.ref.head.name"]: r["time_to_merge"]
        for r in esql(
            """
FROM metrics-githubreceiver.otel-*
| WHERE vcs.change.time_to_merge IS NOT NULL
| STATS time_to_merge = MAX(vcs.change.time_to_merge) BY vcs.ref.head.name
""",
            show=False,
        )
    }
except Exception as exc:
    merge_seconds = {}
    print(f"skipped, no GitHub receiver metrics ({type(exc).__name__})")

open_to_prod = []
for branch, message in sorted(last_commit.items()):
    change = deployed.get(message)
    if change is None or branch not in merge_seconds:
        continue  # never merged, or merged without a deployment
    lead_seconds = (
        ts(change["deployed_at"]) - ts(change["committed_at"])
    ).total_seconds()
    open_to_prod.append((merge_seconds[branch] + lead_seconds) / 60)
if open_to_prod:
    print(
        f"{len(open_to_prod)} pull requests, median open to production {statistics.median(open_to_prod):.1f} minutes"
    )

## 6. Change failure rate and rework rate

`OUTAGE_VERSIONS` in `.env` lists the alerted versions that no deployment caused.

In [ ]:
cfr = esql(
    """
FROM traces-*.otel-*, .alerts-observability.threshold.alerts-default
| WHERE (cicd.pipeline.task.type == "deploy"
        AND deployment.environment.name == "production"
        AND cicd.pipeline.task.run.result == "success")
    OR kibana.alert.rule.name == "checkout failed transaction rate by release"
| STATS deploys = COUNT(*) WHERE cicd.pipeline.task.type == "deploy",
        alerts = COUNT(*) WHERE kibana.alert.rule.name IS NOT NULL
    BY service.version
| WHERE deploys > 0
| STATS deployments = COUNT(*),
        failed_deployments = COUNT(*) WHERE alerts > 0
| EVAL change_failure_rate_pct = ROUND(100.0 * failed_deployments / deployments, 1)
""",
    "Change failure rate",
)

In [ ]:
alerted = esql(
    f"""
FROM traces-*.otel-*, .alerts-observability.threshold.alerts-default
| WHERE (cicd.pipeline.task.type == "deploy"
        AND deployment.environment.name == "production"
        AND cicd.pipeline.task.run.result == "success")
    OR kibana.alert.rule.name == "{RULE_NAME}"
| STATS first_deploy = MIN(@timestamp) WHERE cicd.pipeline.task.type == "deploy",
        first_alert = MIN(kibana.alert.start) WHERE kibana.alert.rule.name IS NOT NULL
    BY service.version
| WHERE first_alert IS NOT NULL
| EVAL minutes_from_deploy_to_alert = ROUND(DATE_DIFF("second", first_deploy, first_alert) / 60.0, 1)
| KEEP service.version, first_deploy, first_alert, minutes_from_deploy_to_alert
| SORT first_alert
""",
    "Alerted versions",
)

OUTAGE_VERSIONS = tuple(
    v for v in os.getenv("OUTAGE_VERSIONS", "b52b78f").split(",") if v
)
CAUSED = {
    r["service.version"]
    for r in alerted
    if not r["service.version"].startswith(OUTAGE_VERSIONS)
}
total = cfr[0]["deployments"]
print(
    f"\nCaused by a deployment: {len(CAUSED)} of {total}, change failure rate {100 * len(CAUSED) / total:.1f}%"
)

In [ ]:
rework = esql(
    """
FROM traces-*.otel-*
| WHERE service.name == "Deliver checkout" AND deployment.environment.name == "production"
| STATS message = MAX(github.head_commit.message),
        attempt = MAX(github.run_attempt),
        deployed = COUNT(*) WHERE cicd.pipeline.task.type == "deploy"
            AND cicd.pipeline.task.run.result == "success"
    BY trace.id
| WHERE deployed > 0
| EVAL rework = attempt > 1 OR message LIKE "Revert*" OR message LIKE "Fix *"
| STATS deployments = COUNT(*),
        rework_deployments = COUNT(*) WHERE rework,
        rework = VALUES(message) WHERE rework
| EVAL rework_rate_pct = ROUND(100.0 * rework_deployments / deployments, 1)
""",
    "Deployment rework rate",
)

## 7. Time to restore

In [ ]:
incidents = esql(
    """
FROM .alerts-observability.threshold.alerts-default
| WHERE kibana.alert.rule.name == "checkout failed transaction rate by release"
| EVAL case.id = kibana.alert.case_ids
| STATS alert_start = MIN(kibana.alert.start),
        alert_end = MAX(kibana.alert.end),
        episodes = COUNT(*)
    BY case.id, service.version
| LOOKUP JOIN .cases ON case.id
| EVAL alert_minutes = ROUND(DATE_DIFF("second", alert_start, alert_end) / 60.0, 1),
       case_minutes = ROUND(case.time_to_resolve / 60.0, 1)
| KEEP alert_start, service.version, episodes, alert_minutes, case_minutes, case.status
| SORT alert_start
""",
    "Incidents",
)

In [ ]:
alert_minutes = [r["alert_minutes"] for r in incidents]
case_minutes = [
    (ts(c["closed_at"]) - ts(c["created_at"])).total_seconds() / 60
    for c in cases
    if c.get("closed_at")
]
caused_alerts = [
    r["alert_minutes"] for r in incidents if r["service.version"] in CAUSED
]
print(
    f"median alert {statistics.median(alert_minutes):.1f} min | median case {statistics.median(case_minutes):.1f} min "
    f"| median alert, deploy-caused only {statistics.median(caused_alerts):.1f} min"
)

deploys = esql(
    """
FROM traces-*.otel-*
| WHERE cicd.pipeline.task.type == "deploy" AND deployment.environment.name == "production"
    AND cicd.pipeline.task.run.result == "success"
| EVAL deploy_end = TO_DATETIME(TO_LONG(@timestamp) + duration / 1000000)
| KEEP service.version, deploy_end
| SORT deploy_end
""",
    show=False,
)
for i, d in enumerate(deploys):
    if d["service.version"] in CAUSED and i + 1 < len(deploys):
        fix = deploys[i + 1]
        print(
            f"bad deploy {d['service.version'][:7]} -> fix {fix['service.version'][:7]}: "
            f"{minutes(d['deploy_end'], fix['deploy_end'])} min"
        )

## 8. Build the dashboard

In [ ]:
existing = requests.get(
    f"{KBN_BASE}/api/dashboards/{DASHBOARD_ID}", headers=KBN_HEADERS, timeout=60
)
if existing.status_code != 200:
    env = {
        **os.environ,
        "KIBANA_URL": KIBANA_URL,
        "KIBANA_SPACE": SPACE,
        "ELASTICSEARCH_API_KEY": API_KEY,
    }
    result = subprocess.run(
        [sys.executable, "dashboard/build_dashboard.py", LAB_START, LAB_END],
        env=env,
        capture_output=True,
        text=True,
    )
    print(result.stdout[:300], result.stderr[-1500:])
    result.check_returncode()
    state["dashboard_created"] = True
    save_state()
print(f"{KBN_BASE}/app/dashboards#/view/{DASHBOARD_ID}")

## 9. Clean up

Deletes only what this notebook created. To remove the lab, see `README.md`.

In [ ]:
deleted = []
if state.get("dashboard_created"):
    requests.delete(
        f"{KBN_BASE}/api/dashboards/{DASHBOARD_ID}", headers=KBN_HEADERS, timeout=60
    ).raise_for_status()
    deleted.append("dashboard")
if state.get("rule_id"):
    found = kbn(
        "GET",
        "/api/cases/_find",
        params={
            "owner": "observability",
            "search": f'"{RULE_NAME}"',
            "searchFields": "title",
            "perPage": 100,
        },
    )["cases"]
    case_ids = [c["id"] for c in found if c["created_at"] >= state["rule_created_at"]]
    if case_ids:
        kbn("DELETE", "/api/cases", params={"ids": json.dumps(case_ids)})
        deleted.append(f"{len(case_ids)} cases")
    kbn("DELETE", f"/api/alerting/rule/{state['rule_id']}")
    deleted.append("rule")
if state.get("data_view_id"):
    kbn("DELETE", f"/api/data_views/data_view/{state['data_view_id']}")
    deleted.append("data view")
STATE_FILE.unlink(missing_ok=True)
print(
    "deleted: "
    + (", ".join(deleted) if deleted else "nothing, this notebook created no objects")
)